In [1]:
import qutip as qt
import numpy as np
import scqubits as scq
import matplotlib.pyplot as plt
from scipy.linalg import expm, sinm, cosm
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

ModuleNotFoundError: No module named 'qutip'

Q1 - C - Q2

Everything is at $\phi_{ext, i}=\phi_0/2$ until told otherwise 

In [ ]:
EC1, EL1, EJ1, flux1 = 1.0, 1.0, 6.0, 0.5

EC2, EL2, EJ2, flux2 = 1.0, 1.0, 5.0, 0.5

ECc, ELc, EJc, fluxc = 1.0, 2.0, 3.0, 0.5

cutoff_f = 70  # fluxonium cutoff dimension

Nf = 5       # retained fluxonium eigenstates truncated_dim

In [ ]:
Jl_vals = np.linspace(0.0, 0.1, 81)   # x-axis Jl12

Jc_vals  = np.linspace(0.0, 0.5, 81)   # y-axis Jc12

In [ ]:
def project_ops(n_sp, phi_sp, V):
    # V: (cutoff, Nf)
    n_red   = V.conj().T @ (n_sp   @ V)
    phi_red = V.conj().T @ (phi_sp @ V)
    return n_red, phi_red

Choose zero flux to be allocated to JJ

In [ ]:
# ---------- build flux=0 scqubits matrices once ----------
def precompute_sc_flux_sparse(EJ, EC, EL, cutoff_f):

    q = scq.Fluxonium(EJ=EJ, EC=EC, EL=EL, flux=0.0, cutoff=cutoff_f)
    H0  = sp.csr_matrix(q.hamiltonian())
    n   = sp.csr_matrix(q.n_operator())
    phi = sp.csr_matrix(q.phi_operator())
    return H0, n, phi

Add $E_L\hat{\varphi}\phi_{ext}$ 

In [ ]:
def single_esys(H0_sp, phi_sp, EL, flux, Nf, v0=None):
    alpha = 2*np.pi*EL*flux
    H_sp = H0_sp + alpha*phi_sp
    evals, evecs = spla.eigsh(H_sp, k=Nf, which="SA", v0=v0)
    idx = np.argsort(evals)
    return evals[idx], evecs[:, idx]   # evecs columns

In [ ]:
d_Hc, d_nc, d_pc = precompute_sc_flux_sparse(EJc, ECc, ELc, cutoff_f)
d_H1, d_n1, d_p1 = precompute_sc_flux_sparse(EJ1, EC1, EL1, cutoff_f)
d_H2, d_n2, d_p2 = precompute_sc_flux_sparse(EJ2, EC2, EL2, cutoff_f)
ec, vc = single_esys(d_Hc, d_pc, ELc, fluxc, Nf, v0=None)
e1, v1 = single_esys(d_H1, d_p1, EL1, flux1, Nf, v0=None)
e2, v2 = single_esys(d_H2, d_p2, EL2, flux2, Nf, v0=None)
# d_qc = scq.Fluxonium(EJ=EJc, EC=ECc, EL=ELc, flux=0.0, cutoff=cutoff_f)
# c_gnd = single_esys(d_Hc, d_pc, ELc, fluxc, Nf, v0=None)[0][0]
# c_gnd

In [ ]:
print(ec,e1,e2)

In [ ]:
def build_2q_blocks_at_flux(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, flux1, fluxc, flux2, Nf, v01=None, v0c=None, v02=None):
    """
    Returns H0_2q, Hc_2q, Hl_2q in the Nf^2 space.
    Then for any (Jc12,Jl12): H2q = H0_2q + Jc12*Hc_2q - Jl12*Hl_2q
    """
    # extract parameters for q1 and q2, they are in cutoff_f * cutoff_f space for now
    H01, n1, phi1 = precompute_sc_flux_sparse(EJ1, EC1, EL1, cutoff_f)
    H0c, nc, phic = precompute_sc_flux_sparse(EJc, ECc, ELc, cutoff_f)
    H02, n2, phi2 = precompute_sc_flux_sparse(EJ2, EC2, EL2, cutoff_f)
    # extract respective eigenvalues/vectors for q1 and q2, truncated to Nf levels
    e1, V1 = single_esys(H01, phi1, EL1, flux1, Nf, v0=v01)
    ec, Vc = single_esys(H0c, phic, ELc, fluxc, Nf, v0=v0c)
    e2, V2 = single_esys(H02, phi2, EL2, flux2, Nf, v0=v02)
    # project operators into this Nf*Nf subspace
    n1_red, phi1_red = project_ops(n1, phi1, V1)
    nc_red, phic_red = project_ops(nc, phic, Vc)
    n2_red, phi2_red = project_ops(n2, phi2, V2)

    #New addition to code: add external flux corrections for interaction terms
    phi1_red = phi1_red + 2*np.pi*flux1*np.eye(Nf)
    phic_red = phic_red + 2*np.pi*fluxc*np.eye(Nf)
    phi2_red = phi2_red + 2*np.pi*flux2*np.eye(Nf)
    # diagonalize Hamiltonians (they include flux allocation correction) and write composite Hamiltonian terms
    # qubit and interaction terms (add coupling strengths later)
    H0_1 = np.kron(np.kron(np.diag(e1), np.eye(Nf)),np.eye(Nf)) 
    H0_c = np.kron(np.kron(np.eye(Nf),np.diag(ec)), np.eye(Nf))
    H0_2 = np.kron(np.kron(np.eye(Nf),np.eye(Nf)), np.diag(e2))
    
    Hc_1c = np.kron(np.kron(n1_red, nc_red), np.eye(Nf))
    Hc_c2 = np.kron(np.kron(np.eye(Nf), nc_red), n2_red)
    Hc_12 = np.kron(np.kron(n1_red, np.eye(Nf)), n2_red)
    Hl_1c = np.kron(np.kron(phi1_red, phic_red), np.eye(Nf))
    Hl_c2 = np.kron(np.kron(np.eye(Nf), phic_red), phi2_red)
    Hl_12 = np.kron(np.kron(phi1_red, np.eye(Nf)), phi2_red)

    # seeds for faster next single-qubit eigsh call
    v01_next = V1[:, 0].real
    v0c_next = Vc[:, 0].real
    v02_next = V2[:, 0].real

    return H0_1, H0_c, H0_2, Hc_1c, Hc_c2, Hc_12, Hl_1c, Hl_c2, Hl_12, v01_next, v0c_next, v02_next#, phi1_red, phic_red, phi2_red

In [ ]:
def energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc12, Jl12, Jc1c, Jl1c, Jcc2, Jlc2, eig_keep=None):
    H0_1, H0_c, H0_2, Hc_1c, Hc_c2, Hc_12, Hl_1c, Hl_c2, Hl_12, _, _, _ = \
        build_2q_blocks_at_flux(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, flux1, fluxc, flux2, Nf, 
                                v01=None, v0c=None, v02=None)
    # dimensions of the subspace
    D = Nf*Nf
    # composite H
    H = H0_1 + H0_c + H0_2 + float(Jc12)*Hc_12 - float(Jl12)*Hl_12 + float(Jc1c)*Hc_1c - float(Jl1c)*Hl_1c +\
     float(Jcc2)*Hc_c2 - float(Jlc2)*Hl_c2

    # dense diagonalization (fast for D <= ~400)
    evals, evecs = np.linalg.eigh(H)

    #keep 40 or less (depending on D) eigenvalues/vectors
    if eig_keep is None:
        eig_keep = min(40, D)
    evals = evals[:eig_keep]
    evecs = evecs[:, :eig_keep]

    # Bare basis indices in ordering |i,j> -> i*Nf + j
    # 0->|000>, 1->|001>, Nf^2->|100>, Nf^2+1->|101>:
    bare_idx = np.array([0, 1, Nf**2, Nf**2 + 1])  # 000,001,100,101

    #keep all the columns 
    overlaps = np.abs(evecs[bare_idx, :])**2  # 4 x eig_keep
    pick = overlaps.argmax(axis=1)

    E000, E001, E100, E101 = evals[pick[0]], evals[pick[1]], evals[pick[2]], evals[pick[3]]
    ZZkHz = 1e6*np.abs(E000 + E101 - E100 - E001)
    return E000, E001, E100, E101, ZZkHz

Coupling notation:\
12 --> Q1-Q2\
1c --> Q1-C\
c2 --> C-Q2

Try only $Jc_{12}$ coupling and compare with Xonium-Xonium design

In [ ]:
ZZ_Conly2 = np.empty(len(Jc_vals))
E000_Conly2 = np.empty(len(Jc_vals))
E100_Conly2 = np.empty(len(Jc_vals))
E001_Conly2 = np.empty(len(Jc_vals))
E101_Conly2 = np.empty(len(Jc_vals))
for i, Jc in enumerate(Jc_vals):
    ZZ_Conly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc,0,0,0,0,0, eig_keep=None)[4]
    E000_Conly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc,0,0,0,0,0, eig_keep=None)[0]
    E100_Conly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc,0,0,0,0,0, eig_keep=None)[2]
    E001_Conly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc,0,0,0,0,0, eig_keep=None)[1]
    E101_Conly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc,0,0,0,0,0, eig_keep=None)[3]

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(Jc_vals, E000_Conly2, label=r"E00")
plt.plot(Jc_vals, E100_Conly2, label=r"E10")
plt.plot(Jc_vals, E001_Conly2, label=r"E01")
plt.plot(Jc_vals, E101_Conly2, label=r"E11")

plt.title("E vs Jc")
plt.ylabel("E, GHz)")
plt.xlabel("Jc, GHz")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(Jc_vals, ZZ_Conly2, label=r"ZZ")

plt.xlabel(r"ZZ vs Jc")
plt.ylabel("ZZ, kHz)")
plt.title("Jc, GHz")
plt.legend()
plt.tight_layout()
plt.show()

These results match with Xonium-Xonium (previous code), for energy spectra, ground energy of the coupler is also added here, which is consistent with X-X results when that is subtracted

Now check where Q1-C and C-Q2 are coupled inductively (same for now), and Q1-Q2 are coupled capacitively

Reminder: Coupling parameter order (Jc12, Jl12, Jc1c, Jl1c, Jcc2, Jlc2)

In [ ]:
ZZ_Lonly2 = np.empty(len(Jl_vals))
E000_Lonly2 = np.empty(len(Jl_vals))
E100_Lonly2 = np.empty(len(Jl_vals))
E001_Lonly2 = np.empty(len(Jl_vals))
E101_Lonly2 = np.empty(len(Jl_vals))
for i, Jl in enumerate(Jl_vals):
    ZZ_Lonly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, 0.1,0,0,Jl,0,Jl, eig_keep=None)[4]
    E000_Lonly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                               flux1, fluxc, flux2, Nf, 0.1,0,0,Jl,0,Jl, eig_keep=None)[0]
    E100_Lonly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                               flux1, fluxc, flux2, Nf, 0.1,0,0,Jl,0,Jl, eig_keep=None)[2]
    E001_Lonly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                               flux1, fluxc, flux2, Nf, 0.1,0,0,Jl,0,Jl, eig_keep=None)[1]
    E101_Lonly2[i] = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, 0.1,0,0,Jl,0,Jl, eig_keep=None)[3]

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(Jl_vals, E000_Lonly2, label=r"E000")
plt.plot(Jl_vals, E100_Lonly2, label=r"E100")
plt.plot(Jl_vals, E001_Lonly2, label=r"E001")
plt.plot(Jl_vals, E101_Lonly2, label=r"E101")

plt.title("E vs Jl")
plt.ylabel("E, GHz)")
plt.xlabel("Jl, GHz")
#plt.xlim(0.09, 0.15)
plt.legend()
plt.tight_layout()
plt.show()

Coupler gnd state energy varies much more with Jl than in Jc case

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(Jl_vals, ZZ_Lonly2, label=r"ZZ")

plt.xlabel(r"ZZ vs Jl")
plt.ylabel("ZZ, kHz)")
plt.title("Jl, GHz")
#plt.xlim(0.09, 0.1)
plt.ylim(0, 1e2)
plt.legend()
plt.tight_layout()
plt.show()

Try this later

In [ ]:
def sweep_flux_fast(EJ1, EC1, EL1, EJ2, EC2, EL2,
                    cutoff_f, Nf, Jc12, Jl12, flux_list):
    pre1 = precompute_sc_flux_sparse(EJ1, EC1, EL1, cutoff_f)
    pre2 = precompute_sc_flux_sparse(EJ2, EC2, EL2, cutoff_f)

    E000s = np.zeros(len(flux_list))
    E001s = np.zeros(len(flux_list))
    E100s = np.zeros(len(flux_list))
    E101s = np.zeros(len(flux_list))
    ZZs  = np.zeros(len(flux_list))

    v01 = v02 = None
    for i, flux in enumerate(flux_list):
        H0_1, H0_c, H0_2, Hc_1c, Hc_c2, Hc_12, Hl_1c, Hl_c2, Hl_12, v01_next, v0c_next, v02_next = \
        build_2q_blocks_at_flux(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                                                flux1, fluxc, flux2, Nf, v01=None, v0c=None, v02=None)
        E000, E001, E100, E101, ZZkHz = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc12, Jl12, Jc1c, Jl1c, Jcc2, Jlc2, eig_keep=None)
        E000s[i], E001s[i], E100s[i], E101s[i], ZZs[i] = E000-E000, E001-E000, E100-E000, E101-E000, ZZkHz

    # plots
    plt.figure(figsize=(6,4))
    plt.plot(flux_list, E000s, label="E00")
    plt.plot(flux_list, E001s, label="E01")
    plt.plot(flux_list, E100s, label="E10")
    plt.plot(flux_list, E101s, label="E11")
    plt.xlabel(r"$\Phi/\Phi_0$")
    plt.ylabel("Energy (your units)")
    plt.legend()
    plt.tight_layout()

    plt.figure(figsize=(6,4))
    plt.plot(flux_list, ZZs)
    plt.xlabel(r"$\Phi/\Phi_0$")
    plt.ylabel(r"$|ZZ|$ (kHz)")
    plt.tight_layout()
    plt.show()

    return E000s, E001s, E100s, E101s, ZZs

Do contour by varying Jc12 and Jl1c and Jlc2

Note the function below assumes same Jl value for both inductive couplings!!!

In [ ]:
def zz_contour_at_flux(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1, fluxc, flux2, Jl12, Jc1c, Jcc2,
                       Jc_list, Jl_list, barmax):
    H0_1, H0_c, H0_2, Hc_1c, Hc_c2, Hc_12, Hl_1c, Hl_c2, Hl_12, v01_next, v0c_next, v02_next = \
        build_2q_blocks_at_flux(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                                                flux1, fluxc, flux2, Nf, v01=None, v0c=None, v02=None)
       
    ZZ = np.zeros((len(Jl_list), len(Jc_list)))  # rows=Jl, cols=Jc

    for i, Jc12 in enumerate(Jc_list):
        for j, Jl in enumerate(Jl_list):
            *_, ZZkHz = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, Jc12, Jl12, Jc1c, Jl, Jcc2, Jl, eig_keep=None)
            ZZ[i, j] = ZZkHz

    # plot
    # Jl_grid, Jc_grid = np.meshgrid(Jl_list, Jc_list)
    # plt.figure(figsize=(6,4))
    # barmin = np.min(ZZ)
    # levels = np.linspace(barmin, barmax, 51)
    # cs = plt.contourf(Jl_grid*1e3, Jc_grid*1e3, ZZ, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
    # plt.xlabel("Jl1c=Jlc2 (MHz)")
    # plt.ylabel("Jc12 (MHz)")
    # plt.title(fr"|ZZ| (kHz) at flux={0.5}")
    # plt.colorbar(cs, label="|ZZ| (kHz)")
    # plt.tight_layout()
    # plt.show()

    return ZZ

Task 0: ZZ map for all fluxes at $\pi/2$. Note that no capacitive coupling between the coupler and qubits is assumed but it does not change the results much.

In [ ]:
#Jl_vals2 = np.linspace(0.0, 0.15, 81)

ZZgrid = zz_contour_at_flux(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1, fluxc, flux2, 0.0, 0.0, 0.0,
                       Jc_vals, Jl_vals2, barmax=1e6)

In [ ]:
Jl_grid, Jc_grid = np.meshgrid(Jl_vals2, Jc_vals)
plt.figure(figsize=(6,4))
barmin = np.min(ZZgrid)
barmax = 30 #np.max(ZZgrid)
levels = np.linspace(barmin, barmax, 51)
cs = plt.contourf(Jl_grid*1e3, Jc_grid*1e3, ZZgrid, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
plt.xlabel("Jl1c=Jlc2 (MHz)")
plt.ylabel("Jc12 (MHz)")
plt.xlim(0, 100)
plt.title(fr"|ZZ| (kHz) at all $\mathrm{{\phi}}_\text{{ext,i}}/\phi_0$={0.5}, Jl12=Jc1c=Jcc2=0")
plt.colorbar(cs, label="|ZZ| (kHz)")
plt.tight_layout()
plt.show()

Task 1: Pick different dc bias for the coupler\
a) $\phi_c=\pi$ \
b) $\phi_c=0$

1.a) $\phi_c=\pi$

In [ ]:
ZZc_pi = zz_contour_at_flux(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1, 1.0, flux2, 0.0, 0.0, 0.0,
                       Jc_vals, Jl_vals2, barmax=1e6)

In [ ]:
Jl_grid, Jc_grid = np.meshgrid(Jl_vals2, Jc_vals)
plt.figure(figsize=(6,4))
barmin = np.min(ZZc_pi)
barmax = 30 #np.max(ZZgrid)
levels = np.linspace(barmin, barmax, 51)
cs = plt.contourf(Jl_grid*1e3, Jc_grid*1e3, ZZc_pi, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
plt.xlabel("Jl1c=Jlc2 (MHz)")
plt.ylabel("Jc12 (MHz)")
plt.xlim(0, 150)
plt.title(fr"|ZZ| (kHz) at $\mathrm{{\phi}}_\text{{c,i}}/\phi_0=\pi$, Jl12=Jc1c=Jcc2=0")
plt.colorbar(cs, label="|ZZ| (kHz)")
plt.tight_layout()
plt.show()

1.b) $\phi_c=0$

In [ ]:
ZZc_0 = zz_contour_at_flux(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1, 0.0, flux2, 0.0, 0.0, 0.0,
                       Jc_vals, Jl_vals2, barmax=1e6)

In [ ]:
Jl_grid, Jc_grid = np.meshgrid(Jl_vals2, Jc_vals)
plt.figure(figsize=(6,4))
barmin = np.min(ZZc_0)
barmax = 30 #np.max(ZZgrid)
levels = np.linspace(barmin, barmax, 51)
cs = plt.contourf(Jl_grid*1e3, Jc_grid*1e3, ZZc_0, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
plt.xlabel("Jl1c=Jlc2 (MHz)")
plt.ylabel("Jc12 (MHz)")
plt.xlim(0, 150)
plt.title(fr"|ZZ| (kHz) at $\mathrm{{\phi}}_\text{{c,i}}/\phi_0=0$, Jl12=Jc1c=Jcc2=0")
plt.colorbar(cs, label="|ZZ| (kHz)")
plt.tight_layout()
plt.show()

Task 2: Add random noise to each external flux $\delta\phi_{ext,i}$, and see how the ZZ coupling varies with coupling map.

In [ ]:
random_flux = np.random.uniform(1e-4, 1.5e-5, size=3)
dflux_1, dflux_c, dflux_2 = random_flux
print(dflux_1, dflux_c, dflux_2)

In [ ]:
ZZ_flux_err = zz_contour_at_flux(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1+dflux_1, fluxc+dflux_c, flux2+dflux_2, 0.0, 0.0, 0.0,
                       Jc_vals, Jl_vals2, barmax=1e6)

In [ ]:
Jl_grid, Jc_grid = np.meshgrid(Jl_vals2, Jc_vals)
plt.figure(figsize=(6,4))
barmin = np.min(ZZ_flux_err)
barmax = 30 #np.max(ZZgrid)
levels = np.linspace(barmin, barmax, 51)
cs = plt.contourf(Jl_grid*1e3, Jc_grid*1e3, ZZ_flux_err, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
plt.xlabel("Jl1c=Jlc2 (MHz)")
plt.ylabel("Jc12 (MHz)")
plt.xlim(0, 100)
plt.title(fr"|ZZ| (kHz) at all $\mathrm{{\phi}}_\text{{ext,i}}/\phi_0$={0.5}+$\delta\phi_i$, Jl12=Jc1c=Jcc2=0")
plt.colorbar(cs, label="|ZZ| (kHz)")
plt.tight_layout()
plt.show()

Task 3: Pick $\phi_c=0$ and add random error to each flux

In [ ]:
ZZ_c0_err = zz_contour_at_flux(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1+dflux_1, 0.0+dflux_c, flux2+dflux_2, 0.0, 0.0, 0.0,
                       Jc_vals, Jl_vals2, barmax=1e6)

In [ ]:
Jl_grid, Jc_grid = np.meshgrid(Jl_vals2, Jc_vals)
plt.figure(figsize=(6,4))
barmin = np.min(ZZ_c0_err)
barmax = 30 #np.max(ZZgrid)
levels = np.linspace(barmin, barmax, 51)
cs = plt.contourf(Jl_grid*1e3, Jc_grid*1e3, ZZ_c0_err, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
plt.xlabel("Jl1c=Jlc2 (MHz)")
plt.ylabel("Jc12 (MHz)")
plt.xlim(0, 150)
plt.title(fr"|ZZ| (kHz) at $\mathrm{{\phi}}_\text{{c,i}}/\phi_0$={0.5}+$\delta\phi_c$, Jl12=Jc1c=Jcc2=0")
plt.colorbar(cs, label="|ZZ| (kHz)")
plt.tight_layout()
plt.show()

---- Ignore the bottom -----

In [ ]:
def zz_contour_at_flux_L_only(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1, fluxc, flux2,
                       Jc_list, Jl_list, barmax):
    H0_1, H0_c, H0_2, Hc_1c, Hc_c2, Hc_12, Hl_1c, Hl_c2, Hl_12, v01_next, v0c_next, v02_next = \
        build_2q_blocks_at_flux(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                                                flux1, fluxc, flux2, Nf, v01=None, v0c=None, v02=None)
       
    ZZ = np.zeros((len(Jl_list), len(Jc_list)))  # rows=Jl, cols=Jc

    for i, Jl_m in enumerate(-Jl_list):
        for j, Jl in enumerate(Jl_list):
            *_, ZZkHz = energies_and_ZZ_from_blocks(EJ1, EJc, EJ2, EC1, ECc, EC2, EL1, ELc, EL2, 
                                flux1, fluxc, flux2, Nf, 0, Jl_m, 0, Jl, 0, Jl, eig_keep=None)
            ZZ[i, j] = ZZkHz

    # plot
    # Jl_grid, Jc_grid = np.meshgrid(Jl_list, Jc_list)
    # plt.figure(figsize=(6,4))
    # barmin = np.min(ZZ)
    # levels = np.linspace(barmin, barmax, 51)
    # cs = plt.contourf(Jl_grid*1e3, Jc_grid*1e3, ZZ, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
    # plt.xlabel("Jl1c=Jlc2 (MHz)")
    # plt.ylabel("Jc12 (MHz)")
    # plt.title(fr"|ZZ| (kHz) at flux={0.5}")
    # plt.colorbar(cs, label="|ZZ| (kHz)")
    # plt.tight_layout()
    # plt.show()

    return ZZ

In [ ]:
ZZgrid_L = zz_contour_at_flux_L_only(EJ1, EC1, EL1, EJc, ECc, ELc, EJ2, EC2, EL2,
                       cutoff_f, Nf, flux1, fluxc, flux2,
                       Jc_vals, Jl_vals, barmax=1e6)

In [ ]:
Jl_grid, Jlm_grid = np.meshgrid(Jl_vals, -Jl_vals)
plt.figure(figsize=(6,4))
barmin = np.min(ZZgrid_L)
barmax = 150#np.max(ZZgrid_L)
levels = np.linspace(barmin, barmax, 51)
cs = plt.contourf(Jl_grid*1e3, Jlm_grid*1e3, ZZgrid_L, levels=levels, cmap="Blues_r", vmin=barmin, vmax=barmax)
plt.xlabel("Jl1c=Jlc2 (MHz)")
plt.ylabel("Jl12 (MHz)")
plt.xlim(0, 20)
plt.ylim(-20,0)
plt.title(fr"|ZZ| (kHz) at flux={0.5}")
plt.colorbar(cs, label="|ZZ| (kHz)")
plt.tight_layout()
plt.show()

ZZ canceling possible with negative and positive inductive couplings, but not positive-positive couplings

- Can you change signs of inductive couplings?
- Play with qubit and coupler parameters